In [1]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


Import Core Libraries and GEE Initialization 

In [2]:
# Cell 2: Import Core Libraries and Load Enriched GBIF Data
import pandas as pd
import geopandas as gpd
import folium # For potential visualization
from configs.regions import kenyan_coast_roi # Your ROI

# Load the environmentally enriched GBIF data locally
gbif_env_enriched_path = os.path.join(project_root, 'data', 'processed', 'gbif_kenya_mangrove_env_enriched.geojson')

if os.path.exists(gbif_env_enriched_path):
    gbif_species_df = gpd.read_file(gbif_env_enriched_path)
    print(f"Loaded {len(gbif_species_df)} enriched GBIF records.")
else:
    print(f"Error: Enriched GBIF data not found at {gbif_env_enriched_path}. Please re-run 06_Data_Integration_Species_Mangroves.ipynb.")
    sys.exit("No enriched GBIF data to analyze.")

print(f"Columns available: {gbif_species_df.columns.tolist()}")

Region of Interest for Kenyan Coast defined.
Loaded 1 enriched GBIF records.
Columns available: ['class', 'decimalLatitude', 'decimalLongitude', 'elevation_m', 'eventDate', 'family', 'gbifID', 'genus', 'is_mangrove', 'kingdom', 'order', 'phylum', 'scientificName', 'species', 'geometry']


Identify Unique Species and Basic Counts

In [3]:
# Cell 3: Identify Unique Species and Basic Counts
print("--- Identifying Key Species in Mangrove Areas ---")

if not gbif_species_df.empty:
    # Get unique scientific names
    unique_species = gbif_species_df['scientificName'].unique()
    print(f"\nUnique species identified in mangrove areas ({len(unique_species)} total):")
    for species_name in unique_species:
        print(f" - {species_name}")

    # Count occurrences per species
    species_counts = gbif_species_df['scientificName'].value_counts().reset_index()
    species_counts.columns = ['scientificName', 'occurrenceCount']
    print("\nSpecies occurrence counts:")
    print(species_counts.head()) # Show top few species

    # Optional: Further filtering for 'true' mangrove species.
    # This requires domain knowledge or an external list of mangrove-associated species.
    # For the MVP, we assume any species found within the buffered mangrove area is relevant.
    # If the single record is a general coastal species, we'll note that.

else:
    print("No species data available for identification.")

--- Identifying Key Species in Mangrove Areas ---

Unique species identified in mangrove areas (1 total):
 - Linckia multifora (Lamarck, 1816)

Species occurrence counts:
                      scientificName  occurrenceCount
0  Linckia multifora (Lamarck, 1816)                1


 Statistical Analysis of Species vs. Environmental Factors

In [4]:
# Cell 4: Statistical Analysis of Species vs. Environmental Factors
print("--- Analyzing Species Distribution vs. Environmental Factors ---")

if not gbif_species_df.empty:
    # Check if there's more than one unique species to make the analysis meaningful.
    # Given we have 1 record, it will only describe that single record's environment.
    num_unique_species = gbif_species_df['scientificName'].nunique()
    if num_unique_species < 1: # Should be >=1 after filtering for existing records
        print("No species data to analyze against environmental factors.")
        sys.exit("Not enough unique species for meaningful analysis.")

    print(f"\nAnalyzing {num_unique_species} unique species against environmental factors:")

    # --- 1. Describe Environmental Conditions for ALL Records ---
    # This will give the min, max, mean, std for each environmental variable for all records.
    # Since we have only 1 record, this will just be the single value.
    env_cols = ['elevation_m', 'mean_annual_temp_C', 'total_annual_prec_mm']
    
    # Check if environmental columns exist
    existing_env_cols = [col for col in env_cols if col in gbif_species_df.columns]
    if not existing_env_cols:
        print("WARNING: No environmental columns found in data for analysis.")
    else:
        print("\nEnvironmental conditions across all species occurrences in mangroves:")
        print(gbif_species_df[existing_env_cols].describe())

        # --- 2. (If >1 species) Describe Environmental Conditions per Species ---
        if num_unique_species > 1:
            print("\nEnvironmental conditions per species (top 5 for brevity):")
            species_env_summary = gbif_species_df.groupby('scientificName')[existing_env_cols].describe()
            print(species_env_summary.head())
        else:
            print(f"\nOnly one unique species ('{gbif_species_df['scientificName'].iloc[0]}') found. Environmental conditions are described above.")


    # --- 3. (Optional for MVP - requires more data) Correlation/Relationship Analysis ---
    # With only 1 (or very few) records, correlation analysis is not statistically meaningful.
    # This would be expanded when you have more species and more occurrences per species.
    # e.g., using linear regression, decision trees, or clustering on larger datasets.

else:
    print("No species data available for environmental analysis.")

--- Analyzing Species Distribution vs. Environmental Factors ---

Analyzing 1 unique species against environmental factors:

Environmental conditions across all species occurrences in mangroves:
       elevation_m
count          1.0
mean           0.0
std            NaN
min            0.0
25%            0.0
50%            0.0
75%            0.0
max            0.0

Only one unique species ('Linckia multifora (Lamarck, 1816)') found. Environmental conditions are described above.
